# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/asraserver06/flyrank-ml-internship-starter/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [9]:
!git clone https://github.com/asraserver06/flyrank-ml-internship-starter.git
%cd flyrank-ml-internship-starter

Cloning into 'flyrank-ml-internship-starter'...
remote: Enumerating objects: 140, done.
remote: Counting objects: 100% (140/140), done.
remote: Compressing objects: 100% (92/92), done.
remote: Total 140 (delta 50), reused 100 (delta 32), pack-reused 0 (from 0)
Receiving objects: 100% (140/140), 1.83 MiB | 9.39 MiB/s, done.
Resolving deltas: 100% (50/50), done.
/content/flyrank-ml-internship-starter/flyrank-ml-internship-starter


In [10]:
!ls data/raw/

content_refresh_anonymized.csv


## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

In [11]:
import pandas as pd
import numpy as np

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")
print(df.shape)

# --- Signal 1: staleness (behind the refresh flags) ---
bins = [0, 30, 90, 180, np.inf]
labels = ['0-30d', '31-90d', '91-180d', '181d+']
df['staleness_bucket'] = pd.cut(df['days_since_last_update'], bins=bins, labels=labels)

signal1_table = df.groupby('staleness_bucket', observed=True).agg(
    n=('content_id', 'count'),
    avg_ctr=('ctr', 'mean'),
    avg_engagement=('engagement_rate', 'mean')
)
print(signal1_table)

(30000, 44)
                      n   avg_ctr  avg_engagement
staleness_bucket                                 
0-30d             20480  0.609021        2.599727
31-90d              175  0.117543        2.134971
91-180d            9171  0.238367        2.406133
181d+               174  3.693276        2.028333


In [12]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


In [13]:
df_pos = df[df['avg_position'] > 0].copy()

pos_bins = [0, 3, 10, 20, 50, np.inf]
pos_labels = ['1-3', '4-10', '11-20', '21-50', '50+']
df_pos['position_bucket'] = pd.cut(df_pos['avg_position'], bins=pos_bins, labels=pos_labels)

signal2_table = df_pos.groupby('position_bucket', observed=True).agg(
    n=('content_id', 'count'),
    avg_ctr=('ctr', 'mean'),
    median_impressions=('impressions_90d', 'median')
)
print(signal2_table)

                     n   avg_ctr  median_impressions
position_bucket                                     
1-3               1141  2.714303                74.0
4-10             11842  0.651045              1184.0
11-20             7273  0.323443               870.0
21-50             7225  0.222345               807.0
50+               1314  0.150784               219.5


Signal 1 — staleness: [table dekh ke likho] CTR/engagement girta hai jaise-jaise days_since_last_update badhta hai → verdict: CONFIRMED (ya jo bhi table dikhaye).

Signal 2 — CTR vs position: Top positions (1-3) mein CTR sabse zyada hai, lekin n aur median_impressions bhi note karo — agar 1-3 bucket ka volume bohot kam hai to caveat likho. Verdict: CONFIRMED / MIXED (table ke hisaab se).

My rule (plain words): Jo content 90+ din se update nahi hua AUR uska CTR apne position-bucket ke average se kam hai, usko refresh ke liye flag karo — kyunke wahi content hai jo already stale hai aur underperform bhi kar raha hai.

Reason codes: stale_low_ctr, stale_ok_ctr, fresh_low_ctr, no_flag

## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

Building the ranked queue: The score is built from two components — how far below the position-bucket average a page's CTR falls (75% weight, since this was the CONFIRMED signal), and normalized staleness on a 0–1 scale (25% weight, since this signal came back MIXED). A higher score means a more urgent flag. Rows with avg_position == 0 (no position data) were excluded upfront. The queue is sorted by score in descending order and written to work/outputs/baseline_action_score.csv.

In [14]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


In [15]:
# score inputs — only pre-outcome, non-label columns
df_pos['ctr_vs_position_avg'] = df_pos['ctr'] - df_pos.groupby('position_bucket', observed=True)['ctr'].transform('mean')

# normalize staleness 0-1
df_pos['staleness_norm'] = (df_pos['days_since_last_update'] / df_pos['days_since_last_update'].max()).clip(0, 1)

# score: higher = more urgent
df_pos['score'] = (
    df_pos['staleness_norm'] * 0.5 +
    (-df_pos['ctr_vs_position_avg']).clip(lower=0) * 0.5
)

def get_reason(row):
    stale = row['days_since_last_update'] >= 90
    low_ctr = row['ctr_vs_position_avg'] < 0
    if stale and low_ctr:
        return 'stale_low_ctr'
    elif stale:
        return 'stale_ok_ctr'
    elif low_ctr:
        return 'fresh_low_ctr'
    return 'no_flag'

df_pos['reason_code'] = df_pos.apply(get_reason, axis=1)

action_map = {
    'stale_low_ctr': 'refresh',
    'stale_ok_ctr': 'monitor',
    'fresh_low_ctr': 'investigate',
    'no_flag': 'no_action'
}
df_pos['action'] = df_pos['reason_code'].map(action_map)

df_ranked = df_pos.sort_values('score', ascending=False).reset_index(drop=True)

import os
os.makedirs("work/outputs", exist_ok=True)
df_ranked.to_csv("work/outputs/baseline_action_score.csv", index=False)
print(f"Wrote {len(df_ranked)} rows to work/outputs/baseline_action_score.csv")

Wrote 28795 rows to work/outputs/baseline_action_score.csv


In [16]:
# score: CTR-vs-position ko zyada weight (CONFIRMED signal), staleness ko kam (MIXED signal)
df_pos['score'] = (
    df_pos['staleness_norm'] * 0.25 +
    (-df_pos['ctr_vs_position_avg']).clip(lower=0) * 0.75
)

df_pos['reason_code'] = df_pos.apply(get_reason, axis=1)
df_pos['action'] = df_pos['reason_code'].map(action_map)

df_ranked = df_pos.sort_values('score', ascending=False).reset_index(drop=True)
df_ranked.to_csv("work/outputs/baseline_action_score.csv", index=False)
print(f"Wrote {len(df_ranked)} rows")

Wrote 28795 rows


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

In [17]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.



*   content_24abafed9707 — action: refresh. Why: 231 days stale, CTR 0% at position 1.3. What would make it wrong: only 4 impressions in 90 days — 0% CTR here is just "0 clicks out of 4 chances," not a real underperformance signal.
content_e2181e471a9c — action: refresh. Why: 211 days stale, CTR 0% at position 3.0. What would make it wrong: only 1 impression — statistically meaningless CTR.
content_e748f498b262 — action: refresh. Why: same client/pattern as above (211 days, position 3.0). What would make it wrong: only 2 impressions — same near-zero-volume problem.
content_5a8e6c869488 — action: refresh. Why: 211 days stale, position 3.0. What would make it wrong: 1 impression total; page may just not be indexed/found yet, not "declining."
content_be9a0395c1e4 — action: refresh. Why: 211 days stale, position 2.0. What would make it wrong: 2 impressions — too little data to call this a CTR problem.
content_a4da7c7eb188 — action: refresh. Why: 211 days stale, position 3.0. What would make it wrong: 1 impression — could be a brand-new or barely-crawled page, not stale content.
content_0edf498ae135 — action: refresh. Why: 211 days stale, position 2.3. What would make it wrong: 3 impressions — same low-volume caveat.
content_4d1ebe33b02d — action: refresh. Why: 151 days stale, position 1.0 (top spot) but CTR 0%. What would make it wrong: only 1 impression — position 1.0 with 1 impression is likely a data artifact, not a real ranking.
content_b51d84226fc9 — action: refresh. Why: same client, same pattern (151 days, position 1.0). What would make it wrong: 1 impression only.
content_a31e10779c01 — action: refresh. Why: 144 days stale, position 2.0. What would make it wrong: 1 impression — no real basis to judge CTR.
content_f86c86d6bd79 — action: refresh. Why: 144 days stale, position 2.0. What would make it wrong: 1 impression, same client cluster — possible duplicate/near-duplicate content skewing the queue.
content_0b5377579ec0 — action: refresh. Why: 144 days stale, position 1.0. What would make it wrong: 1 impression only.
content_a37ce8ddc090 — action: refresh. Why: 106 days stale, position 2.3, CTR 0% — and this one has 1,039 impressions, so the 0% CTR here is real and meaningful, unlike most rows above. What would make it wrong: worth double-checking this isn't a technical issue (broken link, wrong meta title) rather than a content-quality problem — those need a different fix than "refresh."
content_c4f991a58178 — action: refresh. Why: 104 days stale, position 3.0. What would make it wrong: only 2 impressions — low-volume noise again.
content_7d5ad3f9feee — action: refresh. Why: 104 days stale, position 2.7, and 1,916 impressions with 0% CTR — a genuinely strong, high-confidence flag. What would make it wrong: hard to argue this is wrong on the data; worth checking meta title/snippet before assuming it's a "refresh the content" fix vs. a "fix the listing" fix.
content_50dfd64f9e8e — action: refresh. Why: 104 days stale, position 2.4, and 4,446 impressions with 0% CTR — the single strongest, most reliable flag in this list. What would make it wrong: very unlikely to be noise given the volume; only wrong if this is a known/intentional case (e.g. a page being deprecated on purpose).
content_6dd02d2e1d19 — action: refresh. Why: 104 days stale, position 2.3, 278 impressions with 0% CTR — decent volume, moderately reliable. What would make it wrong: 278 is enough to trust the CTR reading, but check if this client's pages generally have low CTR (a client-level pattern, not a content problem).
content_187dc9d91ed0 — action: refresh. Why: 104 days stale, position 1.4, 122 impressions with 0% CTR — a real top-position page with real traffic and zero clicks. What would make it wrong: worth checking the title/snippet directly, since a top-ranked page with any real volume and 0% CTR usually points to a listing problem, not staleness.
content_ad236c4f978e — action: refresh. Why: 104 days stale, position 2.1, 422 impressions with 0% CTR. What would make it wrong: solid volume, so CTR reading is trustworthy — low risk of this being a wrong call.
content_1b74b27f23b9 — action: refresh. Why: 104 days stale, position 2.9, 408 impressions with 0% CTR. What would make it wrong: same as above — reliable given the volume, so unlikely to be a bad flag, but worth confirming the page is actually still live/indexed correctly.
Confidence note har line mein add karo (assignment ne explicitly maanga hai) — jaise "confidence: low (n=1)" ya "confidence: high (n=1916)". Maine already impressions ka zikar kiya har line mein, bas ek short confidence tag daal dena, jaise:
Rows 1–12 (1-4 impressions) → confidence: low
Row 13 (1,039 impressions) → confidence: high
Rows 15–20 (100+ impressions) → confidence: high/medium
Numbering se pehle chhota intro line likh dena, jaise: "Top 20 by score, reviewed individually:"







In [18]:
top20 = df_ranked.head(20)[['content_id', 'client_id', 'score', 'reason_code', 'action',
                              'days_since_last_update', 'ctr', 'avg_position', 'impressions_90d']]
print(top20.to_string())

              content_id          client_id     score    reason_code   action  days_since_last_update  ctr  avg_position  impressions_90d
0   content_24abafed9707  client_8722616204  2.190553  stale_low_ctr  refresh                     231  0.0           1.3                4
1   content_e2181e471a9c  client_d4735e3a26  2.177148  stale_low_ctr  refresh                     211  0.0           3.0                1
2   content_e748f498b262  client_d4735e3a26  2.177148  stale_low_ctr  refresh                     211  0.0           3.0                2
3   content_5a8e6c869488  client_d4735e3a26  2.177148  stale_low_ctr  refresh                     211  0.0           3.0                1
4   content_be9a0395c1e4  client_d4735e3a26  2.177148  stale_low_ctr  refresh                     211  0.0           2.0                2
5   content_a4da7c7eb188  client_d4735e3a26  2.177148  stale_low_ctr  refresh                     211  0.0           3.0                1
6   content_0edf498ae135  client_d

## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

Weak picks: Rows 1–12 of the top 20 are weak picks — each has only 1–4 impressions in the 90-day window, so their 0% CTR reflects near-zero sample size rather than genuine underperformance. A page with 1 impression and 0 clicks tells us almost nothing; it could just as easily convert on the next visit. These rows scored high mainly because they're heavily stale (144–231 days), and staleness alone was already a MIXED signal in Section 1 — so stacking a MIXED signal with an unreliable low-volume CTR reading produces a queue position that isn't well-earned. Rows 13 and 15–20, by contrast, have real impression volume (100–4,446) behind their 0% CTR, making those flags trustworthy.

Leakage check: The score uses only days_since_last_update and ctr (normalized against its position-bucket average) — both of which are metrics available before any decision would be made, not outcomes of a decision. trend_pct, trend_direction, and is_declining_label were never used anywhere in the scoring, bucketing, or feature logic, as required — these are the label-derived columns the data dictionary explicitly flags as leakage risks. No future-window columns (impressions_last_30d, impressions_prev_30d, or anything from the warehouse's forward-looking tables) were used either, since this task only touches the 90-day trailing starter CSV.

In [19]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.